In [ ]:
# ToutAtrac :
# Zygarde50_Forme = % a gérer dans nettoyage + 681 shield et blade

# pd.sum(skipna =True)

### IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import csv

##### Chemin du fichier si local et dans dossier courant 

In [ ]:
import os
chemin_dossier = os.getcwd()
nom_csv = "\\Pokemon_dataset.csv"

chemin_fichier = chemin_dossier + nom_csv
# recupère le dossier courant si csv dans meme dossier
# print(chemin_fichier)

##### Appel classe

In [ ]:
import sys
# abspath transforme chemin relatif (construit avec join) en chemin absolu et os.padir = constante du module os qui contient la chaîne ".." = dossier parent
racine_projet = os.path.abspath(os.path.join(chemin_dossier, os.pardir))

if racine_projet not in sys.path:
    # sys.path = liste de chemins que Python utilise pour rechercher les modules importés
    # Python cherchera d’abord dans ce dossier
    sys.path.insert(0, racine_projet)

from classes.cleanup import PandasImport
from classes.uniformisation import PokemonDf

### Lecture csv et stockage DataFrame

Possible de transposer (.T) et changer axe colonne/ligne dans le pd.read csv

In [ ]:
def lire_csv_bon_encodage(chemin):

    def detect_sep(chemin):
        separateurs = ";,|\t"
        with open(chemin, newline='', encoding='utf-8', errors='ignore') as f:
            sample = f.read(2049) # échantillon des 2049 premiers caractères sur lequel trouver le séparateur
        dialect = csv.Sniffer().sniff(sample, delimiters=separateurs)
        return dialect.delimiter

    separateur = detect_sep(chemin)

    encodings = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]
    for enc in encodings:
        try:
            df = pd.read_csv(chemin, encoding=enc, sep=separateur)
            print(f"Lecture réussie avec l'encodage : {enc}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"Erreur avec {enc} : {e}")
            continue

    raise ValueError("Aucun encodage n'a fonctionné pour ce fichier.")

df = lire_csv_bon_encodage(chemin_fichier)

print(df.head())
# print(f"\n{"_"*75}\n")
# print(df.describe(include="all"))
# print(f"\n{"_"*75}\n")
# print(df.info())
# print(f"\n{"_"*75}\n")

### Premières manips

#####  Transpo, Infos

In [ ]:
df_pokemon = df.T

In [ ]:
print(df_pokemon.shape)
print(f"\n{"_"*75}\n")
print(df_pokemon.describe(include="all"))
print(f"\n{"_"*75}\n")
print(df.info())
print(f"\n{"_"*75}\n")
display(df_pokemon.head(3))
print(f"\n{"_"*75}\n")

##### Copie DataFrame pour pouvoir l'appeler plus loin pour des vérificatons

In [ ]:
df_pokemon_base = df_pokemon.copy()

### Utilitaires

##### Fonction pour voir toutes les lignes

In [ ]:
def see_all(df :pd.DataFrame | pd.Series, limit :int | None = None, shape_on = False):
    with pd.option_context("display.max_rows", limit):
        display(df)
        if shape_on == True:
            print(df.shape)

##### Fonction séparation de colonne

In [ ]:
# séparation sur séparateur spécifique underscore (_)
def split_column(df, col_name, new_cols):
    ind_col = df.columns.get_loc(col_name)

    split_df = df[col_name].str.split("_", n=2, expand=True)[[0, 1]]
    # [[0, 1]] liste qui permet de ne garder que les deux premiers splits pour cas particulier 3 valeurs dans colonne de base et les deux derniers étaient des doublons 
    split_df.columns = new_cols

    if len(new_cols) > 1:
        split_df[new_cols[1]] = split_df[new_cols[1]].fillna("")

    df = df.drop(columns=[col_name])

    df = pd.concat(
        [df.iloc[:, :ind_col], split_df, df.iloc[:, ind_col:]],
        axis=1
    )
    return df

##### Masques

In [ ]:
def choose_mask(df, cols=None, mask=None):
    """
    Retourne un masque 
    """
    if mask == "negativ":
        if cols is None:
            raise ValueError("Il faut donner une `cols`'")
        return (df[cols] < 0).any(axis=1)
  
    if mask == "one_nan":
        if cols is None:
            raise ValueError("Il faut donner une `cols`'")
        return (df[cols].isna().sum(axis=1)) == 1
    
    if mask == "valeur_trop_grande":
        if cols is None:
            raise ValueError("Il faut donner une `cols`'")
        return (df[cols].abs() >= 1000).any(axis=1)
        
    raise ValueError(f"Masque inconnu : {mask}")

##### Fonction voir datas manquantes ou erronées

In [ ]:
def get_missing(df :pd.DataFrame, col :str) -> pd.DataFrame:
    """
    Retourne subset (sous dataframe) avec valeur manquante sur colonnes voulues
    """
    return df[df[col].isna()]

def see_missing(df :pd.DataFrame, col :str, limit :int | None = None) -> None:
    """
    col = total pour voir le total des données manquantes par colonne 
    col = sum_total pour voir le nombre de données manquantes dans tout le df
    sinon choisir la colonne voulue et ensuite la limite de lignes affichées
    + paramètre de see_all a True pour voir le shape du df retourné
    """
    if col == "total":
        subset = df.isna().sum()
    elif col == "sum_total":
        subset = df.isna().sum().sum()
    else:
        subset = get_missing(df, col)

    if isinstance(limit, int) :
        see_all(subset, limit)
    else:
        see_all(subset, limit, True)

In [ ]:
def see_mask(df, mask, limit=None, shape_on=False):
    """
    Affiche uniquement les lignes retenues par un masque.
    """
    subset = df.loc[mask].copy()
    see_all(subset, limit=limit, shape_on=shape_on)
    return 

def get_mask(df, mask):
    """
    Affiche uniquement les lignes retenues par un masque.
    """
    subset = df.loc[mask].copy()
    return subset

In [ ]:
def compare_col_sum_cols(df, col_total, cols_to_sum):
    if col_total not in df.columns:
        raise KeyError(f"Colonne inconnue : {col_total}")

    for col in cols_to_sum:
        if col not in df.columns:
            raise KeyError(f"Colonnes manquantes pour le calcul : {col}")

    expected_total = df[col_total]
    sum_col = df[cols_to_sum].sum(axis=1)

    mask_ok = expected_total.eq(sum_col)
    mask_no = ~mask_ok
    
    invalid_rows = df.loc[mask_no, cols_to_sum + [col_total]].copy()
    invalid_rows["total_attendu"] = sum_col.loc[mask_no]
    invalid_rows["difference"] = invalid_rows[col_total] - invalid_rows["total_attendu"]
    
    # if mask_no.any():
    #     display(df.loc[mask_no])
    return invalid_rows

##### Fonctions calculs pour combler datas manquantes

In [ ]:
def calcul_pourcentage_ratio(df, col_drop, col1, col2): 
    if col_drop not in df.columns:
        return df
    else:
        ind_col = df.columns.get_loc(col_drop)
        new_col = (df[col1] * (df[col_drop] / 100)).round(1)
        df = df.drop(columns=[col_drop])
        df.insert(ind_col, col2, new_col)
    return df 

In [ ]:
def set_mask(df, mask, cols_to_update, transform):
    """
    Applique une transformation sur les lignes sélectionnées par un masque.
    """
    if isinstance(cols_to_update, str):
        cols_to_update = [cols_to_update]

    if callable(transform):
        df.loc[mask, cols_to_update] = transform(df.loc[mask, cols_to_update])
    else:
        df.loc[mask, cols_to_update] = transform

    return df

#### Déclaration de variables

##### Variables des fonctions de la classe cleanup

In [ ]:
do_lower = True
keep_lower = ["IMBD"]
do_capitalize = True
keep_capitalize = ["IMBD"]
del_punctuation = True
keep_punctuation = [":", "_", "°"] # moyen a trouver pour 50% pokemon reste en exception 
del_accent = True
spaces_to_sep = True
sep = '_'

##### Variables contenant labels des colonnes

In [ ]:
col_total = ["total"]
cols_sum_in_total =  ["hp", "attack", "defense", "sp_atck", "sp_def", "speed"]
all_cols = col_total + cols_sum_in_total + ["n°", "name", "generation", "legendary", "type_1", "type_2"]

### Nettoyage

#### HEADER

##### Clean index & renommer labels colonnes

In [ ]:
df_pokemon = df.T # pour reset si je relance la cellule, sinon colonne suivante écrase la précédente

# crée et ajoute une colonne à gauche de la table avec des valeurs allant de 0 à la taille de la dataframe avec un pas de 1 = permet de recréer un index propre (0, 1, 2, ...), ce qui évite un index corrompu ou non aligné. (si pas de données vitales dasn l'index) 
df_pokemon = df_pokemon.reset_index(drop=True)
# argument drop=True: supprime la colonne la plus à gauche avant l'ajout des nouveaux indices

# écrase valeur labels avec les valeurs présentes dans la [ligne donnée]
df_pokemon.columns = df_pokemon.iloc[0]
# supprimer le doublon après copie
df_pokemon = df_pokemon.iloc[1:].reset_index(drop=True)

df_pokemon = df_pokemon.rename(columns={"#": "N°"})

display(df_pokemon)

##### Cleanup

In [ ]:
# Initial list of headers:
*cols, = df_pokemon
print("\nInitial name :\n", cols, "\n")

# Cleaning headers with PandasImport.final_clean (fonction de la classe importée):
pd_tool_clean = PandasImport()

new_label_cols = [
    pd_tool_clean.final_clean(
        str(col_header),
        do_lower=do_lower,
        keep_lower=keep_lower,
        del_punctuation=del_punctuation,
        keep_punctuation=keep_punctuation,
        del_accent=del_accent,
        spaces_to_sep=spaces_to_sep
    )
    for col_header in cols
]
print("\nCleaned columns:\n", new_label_cols)

mapping_header = {old_name: new_name for old_name, new_name in zip(cols, new_label_cols)}
df_pokemon = df_pokemon.rename(columns=mapping_header)
print("\nFinal DataFrame")
display(df_pokemon)

#### DATAFRAME : cleanup

##### clean cellules

In [ ]:
def clean_cell(value):
    if isinstance(value, str):
        return pd_tool_clean.final_clean(
        # fonction classe cleanup
            value,
            do_lower=do_lower,
            keep_lower=keep_lower,
            del_punctuation=del_punctuation,
            keep_punctuation=keep_punctuation,
            del_accent=del_accent,
            spaces_to_sep=spaces_to_sep,
            sep=sep
        )
    return value

df_pokemon[["name", "types", "sp_atk_sp_def"]] = df_pokemon[["name", "types", "sp_atk_sp_def"]].map(clean_cell)

see_all(df_pokemon, 10)

##### Si jamais besoin de gérer les doublons

In [ ]:
# df_pokemon = df_pokemon.drop_duplicates()

##### spliter attaque et défense spéciales

In [ ]:
df_pokemon = split_column(df_pokemon, "sp_atk_sp_def", ["sp_atck", "sp_def"])
display(df_pokemon)

##### format noms et cellules float

In [ ]:
pd_tools = PokemonDf()

df_pokemon = pd_tools.format_columns_names(df_pokemon, ["name", "types"])
display(df_pokemon)

In [ ]:
df_pokemon = pd_tools.format_columns_to_float(df_pokemon, [
"total", "hp", "attack", "defense_of_attack", "sp_atck", "sp_def", "speed", "generation"
])

display(df_pokemon)

##### masque pour vérifier exceptions toutes gérées

In [ ]:
df_pokemon["name"] = df_pokemon["name"].str.title()

pd_tools = PokemonDf()

see_all(df_pokemon[pd_tools.mask_exc_name(df_pokemon)], shape_on=True)

### Tri

##### Trier par numéro pokédex

In [ ]:
df_pokemon = df_pokemon.sort_values("n°", key=lambda numero_pokemon: pd.to_numeric(numero_pokemon, downcast="integer"))

# si beosin reset index une deuxième fois
# df_pokemon = df_pokemon.reset_index(drop=True)
display(df_pokemon)

In [ ]:
display(df_pokemon.sort_index(na_position="first"))

##### Spliter les types

In [ ]:
df_pokemon = split_column(df_pokemon, "types", ["type_1", "type_2"])

display(df_pokemon)

### Calculs sur datas

In [ ]:
# A VERIFIER : une erreur de plus sur df_pokemon par rapport au DataFrame de base
# display(df_pokemon_base[df_pokemon_base[8].isna()].shape)
# display(df_pokemon_base[df_pokemon_base[8].isna()])

see_missing(df_pokemon, col = "total")

##### Calculer la défense sur l'attaque (defense % attaque)

In [ ]:
df_pokemon = calcul_pourcentage_ratio(df_pokemon, "defense_of_attack", "attack", "defense")

display(df_pokemon)

##### Vérifier si total est bon et si données manquantes // 1 fois

In [ ]:
df_compare_total = compare_col_sum_cols(df_pokemon, "total", cols_sum_in_total )

display(df_compare_total)

##### Valeurs négatives

In [ ]:
mask_neg = choose_mask(df_pokemon, cols_sum_in_total, mask="negativ")
# see_neg = see_mask(df_compare_total, mask_neg, shape_on=True)

# ici toutes les valeurs négatives si passent en positif corrige le total donc je les repasse simplement en positif
df_pokemon = set_mask(
    df_pokemon,
    mask_neg,
    cols_to_update=cols_sum_in_total,
    transform=lambda subset: subset.abs()
)

df_compare_total = compare_col_sum_cols(df_pokemon, "total", cols_sum_in_total)
display(df_compare_total)

see_mask(df_pokemon, mask_neg, shape_on=True)

##### Valeurs trop grandes

In [ ]:
mask_val_sup = choose_mask(df_pokemon, cols_sum_in_total, mask="valeur_trop_grande")
# see_mask(df_pokemon, mask=mask_val_sup)

df_pokemon = set_mask(
    df_pokemon,
    mask_val_sup,
    cols_to_update="hp",
    transform=np.nan
)

# beosin de réecraser mon masque pour voir les valeurs après focntion
mask_val_sup_check = choose_mask(df_pokemon, cols_sum_in_total, mask="valeur_trop_grande")
see_mask(df_pokemon, mask_val_sup_check, shape_on=True)

##### Générations manquantes

In [ ]:
# see_all(get_missing(df_pokemon, "generation"), shape_on=True) # = 40

sub_gen_nan = df_pokemon["generation"].isna() # pour avoir un deuxieme bool a comparer avec autre masque, pas un subset comme avec ma fonction 
foward_val = df_pokemon["generation"].ffill()
before_val = df_pokemon["generation"].bfill()

# ------ notna() = bool not missing ------
mask_method_ambiguous = sub_gen_nan & foward_val.notna() & before_val.notna() & (foward_val != before_val)
mask_method_ok = sub_gen_nan & ~mask_method_ambiguous

# see_mask(df_pokemon, mask_method_ok)
# see_mask(df_pokemon, mask_method_ambiguous)

df_pokemon = set_mask(
    df_pokemon,
    mask_method_ok,
    cols_to_update = "generation",
    transform = lambda subset: foward_val.loc[subset.index].fillna(before_val.loc[subset.index])
)

see_missing(df_pokemon, "generation")

In [ ]:
# pour les 5 valeurs ambigües je traite a la main avec une recherche des vraies générations, même pour celles ou j'aurais pu relancer mon bloc précédetn (évite de refaire un deuxième passage)

corrections = {213: 1, 749: 2, 101: 4, 308: 4, 477: 4}  

for idx, gen in corrections.items():
    df_pokemon.loc[idx, "generation"] = gen


see_missing(df_pokemon, "generation")

##### Une valeur manquante (un NaN)

In [ ]:
sub_diff_sur_total = compare_col_sum_cols(df_pokemon, "total", cols_sum_in_total)

mask_one_nan = choose_mask(df_pokemon, cols_sum_in_total, mask="one_nan")
sub_one_nan_total_diff = get_mask(sub_diff_sur_total, mask_one_nan)

valeurs_diff = sub_one_nan_total_diff["difference"]

# dans ma fonction transform(df.loc[mask, cols_to_update]), donc ici subset = df.loc[mask, cols_to_update]
transform_nan_par_diff = lambda subset: subset.apply(
        lambda col: col.where(~col.isna(), valeurs_diff))

df_pokemon = set_mask(
        df_pokemon,
        mask_one_nan,
        cols_to_update = cols_sum_in_total,
        transform = transform_nan_par_diff
)

In [ ]:
see_missing(df_pokemon, col = "total")

##### Deux NaN / pas fait

In [ ]:
check_nan_rest = compare_col_sum_cols(df_pokemon, "total", cols_sum_in_total)
see_all (check_nan_rest, shape_on=True)

In [ ]:
display(df_pokemon[df_pokemon["n°"] == "65"])